# Graflow入門ハンズオンガイド

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GraflowAI/graflow-examples/blob/main/notebooks/langgraph_comparison_tutorial_ja.ipynb)

このノートブックは、Graflowの主要機能をGoogle Colabで実際に動かしながら学べるチュートリアルです。

## このチュートリアルで学べること

| # | トピック | 内容 |
|---|---|---|
| 1 | 最初のグラフ | `@task` + `>>` 演算子で最もシンプルなワークフロー |
| 2 | 並列実行 | `\|` 演算子でDiamondパターン（Fan-out → Fan-in） |
| 3 | データ共有 | Channelによるタスク間データ共有 + 並行安全プリミティブ + 自動キーワード引数解決 |
| 4 | 条件分岐・ループ | `next_task()` / `next_iteration()` / `max_cycles` / `RetryPolicy` |
| 5 | 並列エラーポリシー | Best-effort / At-least-N / Critical ポリシー |
| 6 | ハンズオン | データ分析パイプライン（すべての要素を統合） |
| 7 | Checkpoint/Resume | チェックポイント作成と復元 |
| 8 | カスタムタスクハンドラー | 実行戦略の差し替え（TimingHandler等） |
| 9 | LLM統合 | `inject_llm_client`（Ollama + LiteLLM）によるLLM連携 |

> **Note**: LLM統合（セクション9）は **Ollama**（ローカルLLM）を利用するため、APIキー不要ですべてのセルを実行できます。

## 0. 環境構築

In [ ]:
# pygraphviz のビルドに必要なシステムライブラリをインストール
!apt-get update -qq && apt-get install -y -qq --no-install-recommends graphviz graphviz-dev > /dev/null 2>&1

# ADK（Google ADKエージェント統合）と可視化のオプション依存をインストール
# 全オプションの一覧: https://graflow.ai/docs/getting-started/installation#install-with-all-extras
!pip install -q 'graflow[adk,visualization]>=0.1.6'

In [ ]:
# バージョン確認
import graflow

print(f"Graflow version: {graflow.__version__}")

---

## 1. 最初のグラフを作る — Stateの事前定義は不要

Graflowでは3ステップでワークフローを構築・実行できます:

1. `@task` で関数をタスク化
2. `>>` 演算子で依存関係を定義
3. `ctx.execute()` で実行

**Stateの事前定義（TypedDict）や `compile()` ステップは不要**です。

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("my_pipeline") as ctx:

    @task(inject_context=True)
    def task_a(context: TaskExecutionContext):
        channel = context.get_channel()
        text = channel.get("text", "")
        channel.set("text", text + "a")

    @task(inject_context=True)
    def task_b(context: TaskExecutionContext):
        channel = context.get_channel()
        text = channel.get("text", "")
        channel.set("text", text + "b")

    # >> 演算子で順序を定義
    task_a >> task_b

    # 実行（ret_context=True でチャンネルにアクセス可能）
    _, exec_ctx = ctx.execute("task_a", ret_context=True)
    print(exec_ctx.get_channel().get("text"))  # 'ab'

### Low-Level API: `add_node` / `add_edge` スタイル

`add_node` / `add_edge` による Low-Level API もサポートしています。動的なグラフ構築や細かい制御が必要な場面で使えます。

In [ ]:
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph
from graflow.core.task import TaskWrapper

# LangGraph風にグラフを構築
graph = TaskGraph()

extract = TaskWrapper("extract", func=lambda: print("  extract"), register_to_context=False)
transform = TaskWrapper("transform", func=lambda: print("  transform"), register_to_context=False)
load = TaskWrapper("load", func=lambda: print("  load"), register_to_context=False)

graph.add_node(extract, "extract")
graph.add_node(transform, "transform")
graph.add_node(load, "load")

graph.add_edge("extract", "transform")
graph.add_edge("transform", "load")

# 実行
context = ExecutionContext.create(graph, "extract", max_steps=10)
engine = WorkflowEngine()
engine.execute(context)

---

## 2. 並列実行 — 1行で構造が読める

Graflowでは `>>` が直列、`|` が並列。**Diamond（Fan-out → Fan-in）パターンが1行**で書けます。

```
fetch >> (transform_a | transform_b) >> store
```

In [ ]:
from graflow import task, workflow

with workflow("diamond") as ctx:

    @task
    def fetch():
        print("  データ取得")

    @task
    def transform_a():
        print("  変換A")

    @task
    def transform_b():
        print("  変換B")

    @task
    def store():
        print("  保存")

    # Diamond パターンが1行で表現できる
    fetch >> (transform_a | transform_b) >> store

    ctx.execute("fetch")

### 関数スタイルによる動的タスクリスト構築

`chain()` / `parallel()` 関数を使って、動的にタスクリストを構築することもできます。

In [ ]:
from graflow import parallel, task, workflow

with workflow("dynamic_parallel") as ctx:

    @task
    def start():
        print("  開始")

    # 動的にタスクを生成
    worker_tasks = []
    for i in range(4):

        @task(id=f"worker_{i}")
        def worker(task_id=i):
            print(f"  Worker {task_id} 実行中")

        worker_tasks.append(worker)

    @task
    def aggregate():
        print("  結果を集約")

    # chain + parallel で動的に構築
    start >> parallel(*worker_tasks) >> aggregate

    ctx.execute("start")

---

## 3. タスク間のデータ共有 — Reducerは不要

Graflowでは **Channel（Key-Valueストア）** でタスク間のデータを明示的に読み書きします。「何をいつ更新するか」はタスクの中で制御するため、暗黙的なマージルール（Reducer）は不要です。

並列実行時の競合には、以下の**並行安全プリミティブ**を提供しています（[Concurrency Safety](https://graflow.ai/docs/getting-started/features#concurrency-safety)）:

| プリミティブ | 用途 | MemoryChannel | RedisChannel |
|---|---|---|---|
| `atomic_add(key, amount)` | カウンタ、メトリクス | per-key RLock | INCRBYFLOAT |
| `append(key, value)` | ログ収集、結果の集約（FIFO） | GIL + setdefault | RPUSH |
| `prepend(key, value)` | 優先キュー、LIFO スタック | GIL + setdefault | LPUSH |
| `lock(key)` | 条件付き更新、複数キーの整合操作 | per-key RLock | 分散ロック |

RedisChannel バックエンドを利用した場合は、Redisのシングルスレッド動作により、複数ワーカーからの読み書きはRedis側でリクエストが直列化されます。アプリケーション側でロックや排他制御は不要です。

### Channel の基本: `set` / `get`

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("channel_demo") as ctx:

    @task(inject_context=True)
    def producer(context: TaskExecutionContext):
        channel = context.get_channel()
        channel.set("config", {"batch_size": 100})
        channel.set("counter", 1)
        print(f"  Producer: config={channel.get('config')}, counter={channel.get('counter')}")

    @task(inject_context=True)
    def consumer(context: TaskExecutionContext):
        channel = context.get_channel()
        config = channel.get("config")  # {"batch_size": 100}
        counter = channel.get("counter")  # 1
        channel.set("counter", counter + 1)  # 明示的に更新
        print(f"  Consumer: config={config}, counter={channel.get('counter')}")

    producer >> consumer
    ctx.execute("producer")

### 自動キーワード引数解決

`inject_context` すら不要！タスクの引数名がチャンネルのキーと自動的にマッチングされます。

仕組み: タスク関数のシグネチャを `inspect.signature()` で解析し、引数名に一致するキーがChannelに存在すれば自動的に値を注入します。デフォルト値を持つ引数はChannelにキーがなければデフォルト値が使われ、デフォルト値のない引数はChannelにキーがなければエラーになります。

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("auto_resolve") as ctx:

    @task(inject_context=True)
    def setup(context: TaskExecutionContext):
        channel = context.get_channel()
        channel.set("user_name", "Alice")
        channel.set("user_city", "Tokyo")

    @task
    def greet(user_name: str, user_city: str = "Unknown"):
        # inject_context 不要！引数名がチャンネルのキーと自動一致
        print(f"  Hello, {user_name} from {user_city}!")

    setup >> greet
    ctx.execute("setup")

### TypedChannel — 型安全なデータ共有

`TypedDict` でスキーマを定義し、`get_typed_channel()` で型付きチャンネルを取得すると、IDEの**コード補完やタイプチェック**が有効になります。キー名のタイプミスや型の不整合をコーディング時に検出でき、大規模なワークフローでも安全にデータを扱えます。

In [ ]:
from graflow import TaskExecutionContext, task, workflow
from typing import TypedDict

class UserProfile(TypedDict):
    user_id: str
    name: str
    email: str

with workflow("typed_channel_demo") as ctx:

    @task(inject_context=True)
    def collect_user(context: TaskExecutionContext):
        profile_ch = context.get_typed_channel(UserProfile)
        profile_ch.set(
            "current_user",
            {
                "user_id": "u_123",
                "name": "Alice",
                "email": "alice@example.com",
            },
        )
        print("  ユーザー情報を保存しました")

    @task(inject_context=True)
    def display_user(context: TaskExecutionContext):
        profile_ch = context.get_typed_channel(UserProfile)
        user = profile_ch.get("current_user")
        print(f"  ユーザー: {user['name']} ({user['email']})")

    collect_user >> display_user
    ctx.execute("collect_user")

---

## 4. 条件分岐とループ — 事前定義不要の動的制御

Graflowでは、分岐やループを**実行時に動的に制御**できます。事前にすべてのパスを定義する必要はありません。

- `next_task()`: 実行時に新しいタスクを追加（動的分岐）
- `next_iteration()`: 自分自身を再実行（リトライ、収束ループ）

### `next_iteration()` — リトライ（自分自身を再実行）

In [ ]:
from graflow import TaskExecutionContext, task, workflow
import random

with workflow("retry_demo") as ctx:

    @task(inject_context=True)
    def process_data(context: TaskExecutionContext):
        channel = context.get_channel()
        attempt = channel.get("attempt", 0) + 1
        channel.set("attempt", attempt)

        # ランダムなスコアをシミュレーション（試行回数が増えると成功しやすくなる）
        score = random.random() * (0.5 + attempt * 0.15)
        score = min(score, 1.0)
        print(f"  試行 {attempt}: score = {score:.2f}")

        if score < 0.8 and attempt < 5:
            # リトライ: 自分自身を再実行
            print("  → スコアが低いためリトライ")
            context.next_iteration()
        else:
            channel.set("final_score", score)
            print(f"  → 成功! 最終スコア: {score:.2f}")

    @task
    def finalize(final_score: float):
        print(f"  完了: 最終スコア = {final_score:.2f}")

    process_data >> finalize

    random.seed(42)
    ctx.execute("process_data")

### `next_task()` — 動的分岐（新しいタスクを実行時に生成）

`next_task()` は**新しいタスクの生成＋グラフへの追加**を実行時に行います。ファイル数やAPIレスポンスなど、実行時にしか判明しない情報に基づいてタスクを動的に生成できます。

In [ ]:
from graflow import TaskExecutionContext, task, workflow
from graflow.core.task import TaskWrapper

with workflow("dynamic_fanout") as ctx:

    @task(inject_context=True)
    def discover_files(context: TaskExecutionContext):
        """実行時にファイル数が判明し、それに応じてタスクを動的生成"""
        files = ["report.csv", "users.json", "logs.txt"]
        print(f"  {len(files)} 個のファイルを発見")

        # ファイル数に応じてタスクを動的生成（Fan-out）
        for file in files:
            context.next_task(
                TaskWrapper(
                    f"process_{file}",
                    lambda f=file: print(f"  処理中: {f}"),
                )
            )

    discover_files  # 起点タスクのみ定義。後続は実行時に動的生成

    ctx.execute("discover_files")

### 動的制御メソッド一覧

| メソッド | 動作 | 用途 |
|---|---|---|
| `next_task(task)` | 新しいタスクを追加 | 動的分岐、Fan-out |
| `next_task(task, goto=True)` | 既存タスクにジャンプ（後続スキップ） | 早期脱出、エラーハンドリング |
| `next_iteration()` | 自分自身を再実行 | リトライ、収束ループ |
| `terminate_workflow()` | 正常終了 | 早期完了 |
| `cancel_workflow(reason)` | 異常終了 | エラー時のキャンセル |

### `max_cycles` — イテレーション回数の制御

`next_iteration()` による手動ループに加え、`@task(max_cycles=N)` でイテレーション上限を宣言的に設定できます。`ctx.can_iterate()` で残りサイクルがあるかを確認し、`ctx.cycle_count` で現在のサイクル番号（1始まり）を取得できます。

In [ ]:
from graflow import task
from graflow.core.context import ExecutionContext, TaskExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph

graph = TaskGraph()

@task(inject_context=True, max_cycles=20)
def optimizer(ctx: TaskExecutionContext, data=None):
    loss = (data or {}).get("loss", 1.0) * 0.5
    print(f"  cycle {ctx.cycle_count}: loss={loss:.4f}")
    if loss < 0.05:
        print(f"  Converged at cycle {ctx.cycle_count}")
        return
    if ctx.can_iterate():
        ctx.next_iteration({"loss": loss})

graph.add_node(optimizer, "optimizer")
WorkflowEngine().execute(
    ExecutionContext.create(graph, start_node="optimizer")
)

### `RetryPolicy` — 失敗時の自動リトライ

`@task` デコレータで `RetryPolicy` を直接指定でき、`jitter`（ランダム化）や `max_interval`（上限キャップ）といったオプションも利用できます。`default_max_retries` でワークフロー全体のデフォルトも設定可能です。

In [ ]:
from graflow import RetryPolicy, task
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph
from graflow.exceptions import GraflowRuntimeError

graph = TaskGraph()
attempts = 0

@task(
    retry_policy=RetryPolicy(
        max_retries=3,
        initial_interval=0.1,  # short for demo
        backoff_factor=2.0,    # 0.1s → 0.2s → 0.4s
    ),
)
def flaky_service():
    global attempts
    attempts += 1
    print(f"  attempt {attempts}")
    if attempts < 3:
        raise ConnectionError(f"timeout (attempt {attempts})")
    return "success"

graph.add_node(flaky_service, "flaky_service")
context = ExecutionContext.create(graph, start_node="flaky_service")
WorkflowEngine().execute(context)
print(f"  Recovered after {attempts} attempts, result: {context.get_result('flaky_service')}")

---

## 5. 並列グループのエラーポリシー

Graflowでは**並列グループ単位で柔軟にエラー制御**が可能です。

| ポリシー | 動作 | 適用例 |
|---|---|---|
| **Strict**（デフォルト） | 全タスク成功必須 | 金融取引、データ検証 |
| **Best-effort** | 部分成功で続行 | 通知送信、オプション処理 |
| **At-least-N** | 最小成功数を指定 | マルチリージョンデプロイ |
| **Critical** | 重要タスクのみ判定 | 必須+オプションの混在 |

In [ ]:
from graflow import task, workflow
from graflow.coordination.executor import CoordinationBackend
from graflow.core.handlers.group_policy import (
    BestEffortGroupPolicy,
)

with workflow("error_policy_demo") as ctx:

    @task
    def send_email():
        print("  Email 送信成功")

    @task
    def send_sms():
        raise RuntimeError("SMS送信失敗!")  # わざと失敗させる

    @task
    def send_slack():
        print("  Slack 送信成功")

    @task
    def done():
        print("  通知処理完了（一部失敗しても続行）")

    # BestEffortPolicy: 一部失敗しても続行
    (send_email | send_sms | send_slack).with_execution(
        backend=CoordinationBackend.THREADING,
        policy=BestEffortGroupPolicy(),
    ) >> done

    ctx.execute("send_email")

---

## 6. ハンズオン 〜データ分析パイプライン〜

これまでの要素を組み合わせた実践的なパイプラインです。

- `@task` デコレータと `>>` / `|` 演算子
- Channel によるタスク間データ共有
- 自動キーワード引数解決（`generate_report` の引数がチャンネルから自動取得）
- Diamond パターン（Fan-out → Fan-in）

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("data_analysis") as ctx:

    @task(inject_context=True)
    def fetch_data(context: TaskExecutionContext):
        """データを取得してチャンネルに格納"""
        data = {
            "sales": [100, 200, 150, 300, 250],
            "costs": [50, 80, 60, 120, 100],
        }
        channel = context.get_channel()
        channel.set("raw_data", data)
        print(f"📥 データ取得: {len(data['sales'])}件")

    @task(inject_context=True)
    def analyze_sales(context: TaskExecutionContext):
        """売上分析（並列タスク1）"""
        channel = context.get_channel()
        sales = channel.get("raw_data")["sales"]
        total = sum(sales)
        channel.set("sales_total", total)
        print(f"📊 売上分析: 合計={total}, 平均={total / len(sales)}")

    @task(inject_context=True)
    def analyze_costs(context: TaskExecutionContext):
        """コスト分析（並列タスク2）"""
        channel = context.get_channel()
        costs = channel.get("raw_data")["costs"]
        total = sum(costs)
        channel.set("cost_total", total)
        print(f"💰 コスト分析: 合計={total}, 平均={total / len(costs)}")

    @task
    def generate_report(sales_total: int, cost_total: int):
        """分析結果を統合してレポート生成（自動キーワード引数解決）"""
        profit = sales_total - cost_total
        margin = (profit / sales_total) * 100
        print("\n📝 === 分析レポート ===")
        print(f"   売上合計: {sales_total}")
        print(f"   コスト合計: {cost_total}")
        print(f"   利益: {profit} （利益率: {margin:.1f}%）")

    # ワークフロー定義（1行で構造が読める）
    fetch_data >> (analyze_sales | analyze_costs) >> generate_report

    ctx.execute("fetch_data")

---

## 7. Checkpoint/Resume — 中断と再開

長時間ワークフローでは、途中経過を保存して後から再開できることが重要です。Graflowでは**ユーザーが重要なポイントを選んで保存**できるため、効率的です。

**ポイント:**
- `task_ctx.checkpoint(path=...)` でチェックポイントを作成
- `CheckpointManager.resume_from_checkpoint(path)` で復元
- タスクの実行結果（Channel）も含めて保存・復元される

In [ ]:
import os
import tempfile

from graflow import TaskExecutionContext, task, workflow
from graflow.core.checkpoint import CheckpointManager
from graflow.core.engine import WorkflowEngine

# 一時ディレクトリにチェックポイントを保存
with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, "demo_checkpoint")

    # === Part 1: ワークフロー実行 + チェックポイント作成 ===
    with workflow("checkpoint_demo") as wf:

        @task
        def step_1() -> str:
            print("Step 1: データ準備")
            return "data_prepared"

        @task(inject_context=True)
        def step_2(task_ctx: TaskExecutionContext) -> str:
            print("Step 2: 処理実行")
            # このタスク完了後にチェックポイントを作成
            task_ctx.checkpoint(path=checkpoint_path, metadata={"stage": "processing_done"})
            return "data_processed"

        @task
        def step_3() -> str:
            print("Step 3: 最終処理")
            return "completed"

        step_1 >> step_2 >> step_3

        # 実行（ret_context=True でコンテキストも取得）
        _result, context = wf.execute("step_1", ret_context=True)

    print(f"\nチェックポイント保存先: {context.last_checkpoint_path}")
    print(f"実行ステップ数: {context.steps}")

    # === Part 2: チェックポイントから復元 ===
    print("\n--- チェックポイントから復元 ---")
    restored_ctx, metadata = CheckpointManager.resume_from_checkpoint(
        context.last_checkpoint_path
    )

    print(f"復元されたタスク: {list(restored_ctx.completed_tasks)}")
    print(f"step_1 の結果: {restored_ctx.get_result('step_1')}")
    print(f"step_2 の結果: {restored_ctx.get_result('step_2')}")

    # 復元したコンテキストで続行（step_3 から）
    print("\n--- 復元後に実行を続行 ---")
    engine = WorkflowEngine()
    final_result = engine.execute(restored_ctx)

    print(f"\nstep_3 の結果: {restored_ctx.get_result('step_3')}")
    print(f"最終結果: {final_result}")

---

## 8. カスタムタスクハンドラー — 実行戦略の差し替え

Graflowでは、タスクの**実行方法**をハンドラーで制御できます。デフォルトはプロセス内実行（`direct`）ですが、
カスタムハンドラーを作成して `@task(handler="...")` で指定することで、ロギング・タイミング計測・リモート実行など
自由な実行戦略を差し込めます。

**ハンドラーの作り方:**
1. `TaskHandler` を継承して `execute_task()` を実装
2. `WorkflowEngine` にハンドラーを登録
3. `@task(handler="名前")` で利用

> **Note:** Graflowには `DockerTaskHandler` も組み込まれており、タスクをDockerコンテナ内で隔離実行できます。
> Colab環境ではDockerが利用できないためここではカスタムハンドラーの例を示しますが、
> 詳細は [Docker Task Handler](https://graflow.ai/docs/tutorial/advanced/task-handlers#docker-task-handler) を参照してください。

In [ ]:
import time

from graflow import task, workflow
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.handler import TaskHandler
from graflow.core.task import Executable

# --- カスタムハンドラー: 実行時間を計測する ---
class TimingHandler(TaskHandler):
    """タスクの実行時間を計測するカスタムハンドラー"""

    def execute_task(self, task: Executable, context: ExecutionContext):
        task_id = task.task_id
        start = time.perf_counter()
        try:
            result = task.run()
            elapsed = time.perf_counter() - start
            print(f"  [TimingHandler] {task_id} completed in {elapsed:.3f}s")
            context.set_result(task_id, result)
            return result
        except Exception as e:
            context.set_result(task_id, e)
            raise

# --- ワークフロー定義 ---
with workflow("handler_demo") as ctx:

    @task(handler="direct")  # デフォルトハンドラー
    def fetch_data() -> dict:
        print("fetch_data: データ取得中...")
        return {"values": [1, 2, 3, 4, 5]}

    @task(handler="timing")  # カスタムハンドラーで実行
    def process_data(fetch_data: dict) -> float:
        print("process_data: 計算中...")
        time.sleep(0.1)  # 重い処理をシミュレート
        return sum(fetch_data["values"]) / len(fetch_data["values"])

    @task(handler="timing")  # カスタムハンドラーで実行
    def format_result(process_data: float) -> str:
        print("format_result: フォーマット中...")
        return f"平均値: {process_data:.1f}"

    fetch_data >> process_data >> format_result

    # エンジンにカスタムハンドラーを登録して実行
    engine = WorkflowEngine()
    engine.register_handler("timing", TimingHandler())

    exec_context = ExecutionContext.create(ctx.graph, "fetch_data")
    result = engine.execute(exec_context)

print(f"\n結果: {result}")

---

## 9. LLM統合 — `inject_llm_client`

Graflowは [LiteLLM](https://docs.litellm.ai/) を通じて、Ollama / OpenAI / Claude / Gemini など100以上のLLMプロバイダーを統一APIで利用できます。
このチュートリアルでは **Ollama** を使ってローカルLLMで実行します。

#### Colab で使えるモデル候補

Google Colab 無料枠は **CPU（RAM 約12.7GB）** または **T4 GPU（VRAM 15GB）** が利用できます。
以下は Ollama で利用可能な代表的モデルです。

| モデル | パラメータ数 | モデルサイズ | コンテキスト長 | 量子化 | 備考 |
|--------|------------|------------|--------------|--------|------|
| [gemma4:e4b](https://ollama.com/library/gemma4:e4b) | 4.5B（実効） | 9.6 GB | 128K | Q4_K_M | テキスト・画像・音声対応。**本チュートリアルで使用** |
| [llama3.1:8b](https://ollama.com/library/llama3.1:8b) | 8.0B | 4.9 GB | 128K | Q4_K_M | Meta 製。軽量で高性能なテキスト生成モデル |
| [gpt-oss:20b](https://ollama.com/library/gpt-oss) | 20B（MoE） | 14 GB | 128K | MXFP4 | T4 GPU 推奨。CPU RAM 12.7GB では不足 |

> **💡 推奨:** Colab のランタイムタイプを **T4 GPU** に変更すると、推論が大幅に高速化されます（メニュー「ランタイム」→「ランタイムのタイプを変更」→ T4 GPU を選択）。T4 の VRAM 15GB なら上記すべてのモデルが動作します。

> **本チュートリアルでは `gemma4:e4b` を使用します。** CPU / T4 GPU どちらでも動作します。
> 別モデルを試す場合は、以下のモデルダウンロードセルと `LLMClient(model=...)` のモデル名を変更してください。

### 方式1: `inject_llm_client`（LiteLLM統合）

`@task(inject_llm_client=True)` を指定すると、タスクの第1引数に `LLMClient` が自動注入されます。
LLMClientは [LiteLLM](https://docs.litellm.ai/) のラッパーで、OpenAI / Claude / Gemini など100以上のLLMプロバイダーを統一APIで利用できます。

**ポイント:**
- `LLMClient` はワークフロー内で**シングルトン**（全タスクで同一インスタンスを共有）
- デフォルトモデルを設定しつつ、呼び出しごとに別モデルを指定可能
- このチュートリアルでは **Ollama**（ローカルLLM）を使用するため、APIキー不要で動作します
- `inject_llm_client=True` 指定時に `LLMClient` が未登録の場合、デフォルトモデル（`GRAFLOW_LLM_MODEL` 環境変数で変更可能）で自動生成

In [ ]:
!pip install -q litellm

# --- Ollama のセットアップ（Google Colab 用） ---
# Colab 環境ではローカルに Ollama をインストールして起動します。
# APIキー不要でLLMを利用できます。

import os
import time

# zstd のインストール（Ollama インストーラーが必要とします）
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1

# CUDAドライバーのインストール（T4 GPU ランタイムで Ollama が GPU を認識するために必要）
!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get install -y -qq cuda-drivers > /dev/null 2>&1

# Ollama のインストール
!curl -fsSL https://ollama.com/install.sh | sh

# Ollama サーバーをバックグラウンドで起動
!nohup ollama serve &
time.sleep(3)  # サーバー起動を待機

# LiteLLM が Ollama を使うための設定
os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"

print("Ollama server ready!")

In [ ]:
# モデルのダウンロード（初回のみ時間がかかります）
!ollama pull gemma4:e4b
# !ollama pull llama3.1:8b

In [ ]:
from graflow import task, workflow
from graflow.core.context import ExecutionContext, TaskExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.llm.client import LLMClient

with workflow("llm_client_demo") as ctx:
    # LLMClient を作成（全タスクで共有）
    llm_client = LLMClient(model="ollama/gemma4:e4b", enable_tracing=False)

    @task(inject_context=True, inject_llm_client=True)
    def summarize(ctx: TaskExecutionContext, llm_client: LLMClient):
        """LLMClient が自動注入され、結果を summary としてチャネルに格納"""
        result = llm_client.completion_text(
            messages=[{"role": "user", "content": "Pythonを一言で説明してください"}],
        )
        ctx.get_channel().set("summary", result)

    @task(inject_llm_client=True)
    def translate(llm_client: LLMClient, summary: str) -> str:
        """Channel から summary を自動解決 + LLMClient 注入"""
        return llm_client.completion_text(
            messages=[{"role": "user", "content": f"Translate to English: {summary}"}],
        )

    # summarize が Channel に "summary" を格納し、
    # translate の引数 summary に自動解決される
    summarize >> translate

    # ExecutionContext に llm_client を渡して実行
    exec_context = ExecutionContext.create(ctx.graph, "summarize", llm_client=llm_client)
    result = WorkflowEngine().execute(exec_context)

print(f"翻訳結果: {result}")

#### LLMClient の自動生成

`llm_client` を `ExecutionContext.create()` に渡さなくても、`inject_llm_client=True` のタスクが実行されると **デフォルトの LLMClient が自動生成**されます。
デフォルトモデルは環境変数 `GRAFLOW_LLM_MODEL` で変更可能です。ここでは Ollama の `gemma4:e4b` を指定します。

In [ ]:
from graflow import task, workflow
from graflow.llm.client import LLMClient

import os
os.environ["GRAFLOW_LLM_MODEL"] = "ollama/gemma4:e4b"

with workflow("llm_auto_demo") as ctx:
    # llm_client を明示的に渡さない → GRAFLOW_LLM_MODEL で指定したモデルが自動生成される

    @task(inject_llm_client=True)
    def ask(llm_client: LLMClient) -> str:
        print(f"Auto-created model: {llm_client.model}")
        return llm_client.completion_text(
            messages=[{"role": "user", "content": "What is 1+1? Answer in one word."}],
        )

    result = ctx.execute("ask")

print(f"回答: {result}")

#### 呼び出しごとのモデル切り替え

同一タスク内で、呼び出しごとに異なるモデルを使い分けることも可能です。

In [ ]:
from graflow import task, workflow
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.llm.client import LLMClient

with workflow("llm_model_switch") as ctx:
    llm_client = LLMClient(model="ollama/gemma4:e4b", enable_tracing=False)

    @task(inject_llm_client=True)
    def multi_model(llm_client: LLMClient) -> dict:
        # デフォルトモデル（ollama/gemma4:e4b）で処理
        quick = llm_client.completion_text(
            messages=[{"role": "user", "content": "Say hello in Japanese"}],
        )
        # 別モデルを指定して呼び出し
        detailed = llm_client.completion_text(
            messages=[{"role": "user", "content": "Explain quantum computing in one sentence"}],
            model="ollama/gemma4:e4b",  # 明示的にモデルを指定（同じモデルでもOK）
        )
        return {"quick": quick, "detailed": detailed}

    # ExecutionContext に llm_client を渡して実行
    exec_context = ExecutionContext.create(ctx.graph, "multi_model", llm_client=llm_client)
    result = WorkflowEngine().execute(exec_context)

print(f"結果1: {result['quick']}")
print(f"結果2: {result['detailed']}")

### 方式2: `inject_llm_agent`（Google ADK統合）

`@task(inject_llm_agent="agent_name")` を指定すると、`ExecutionContext` に登録された `LLMAgent` がタスクの引数に自動注入されます。
Google ADK の `LlmAgent` をラップした `AdkLLMAgent` を使うことで、**ツール呼び出し（ReAct パターン）** を含むエージェントをワークフローに組み込めます。

In [ ]:
from google.adk.agents import LlmAgent
from graflow import task, workflow
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.llm.agents import AdkLLMAgent, LLMAgent


# ADK ツール定義（関数がそのままツールになる）
def get_weather(city: str) -> dict:
    """指定された都市の天気を取得する。"""
    # デモ用のダミーデータ
    weather_data = {
        "Tokyo": {"temp": 22, "condition": "晴れ"},
        "Osaka": {"temp": 25, "condition": "曇り"},
        "Sapporo": {"temp": 15, "condition": "雨"},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "不明"})
    return {"city": city, "temp_celsius": data["temp"], "condition": data["condition"]}


with workflow("adk_agent_demo") as ctx:
    # ツール付き ADK エージェントを作成
    adk_agent = LlmAgent(
        name="weather_agent",
        model="ollama/gemma4:e4b",  # model="gemini-2.5-flash",
        instruction="あなたは天気予報アシスタントです。get_weather ツールを使って回答してください。",
        tools=[get_weather],
    )

    @task(inject_llm_agent="weather_agent")
    def ask_weather(llm_agent: LLMAgent) -> str:
        """エージェントがツールを呼び出して天気情報を取得・要約"""
        result = llm_agent.run("東京と札幌の天気を教えてください")
        return result["output"]

    # ExecutionContext を作成し、エージェントを登録
    exec_context = ExecutionContext.create(ctx.graph, "ask_weather")
    agent = AdkLLMAgent(adk_agent, app_name=exec_context.session_id, enable_tracing=False)
    exec_context.register_llm_agent("weather_agent", agent)

    result = WorkflowEngine().execute(exec_context)

print(f"天気レポート:\n{result}")

---

## まとめ

このチュートリアルでは、Graflowの主要機能を一通り体験しました。

### 機能一覧

| 機能 | 概要 |
|---|---|
| **グラフ定義** | `>>` / `\|` 演算子（1行で構造が読める）+ Low-Level API も対応 |
| **データ共有** | Channel（Key-Value） + 自動キーワード引数解決 |
| **条件分岐** | `next_task()` / `next_iteration()`（実行時動的） |
| **チェックポイント** | ユーザー制御（任意タイミングで保存・復元） |
| **並列エラー制御** | 4種の組み込みポリシー + カスタム |
| **タスクハンドラー** | direct / docker / カスタム |
| **LLM統合** | LiteLLM + Google ADK エージェント |
| **トレーシング** | Langfuse（OSS） + OpenTelemetry |
| **分散実行** | Redisベースワーカーで水平スケール |
| **設計方針** | Define-by-Run（実行しながらグラフ構築） |

詳細は [Graflow公式ドキュメント](https://graflow.ai/docs) を参照してください。